In [15]:
!pip install pandas matplotlib seaborn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import os
print("Current directory:", os.getcwd())
print("Files here:", os.listdir())

Current directory: c:\Users\camil\Desktop\Python project Egypt
Files here: ['.venv', 'ACLED_egypt2011.csv', 'bella_copia.ipynb']


In [17]:
import os
print("Current working directory:", os.getcwd())
print("\nAll files in current directory:")
for file in os.listdir():
    print(f"  - {file}")

Current working directory: c:\Users\camil\Desktop\Python project Egypt

All files in current directory:
  - .venv
  - ACLED_egypt2011.csv
  - bella_copia.ipynb


In [18]:
import os
print("Current working directory:", os.getcwd())
print("\nAll files in current directory:")
for file in os.listdir():
    print(f"  - {file}")

Current working directory: c:\Users\camil\Desktop\Python project Egypt

All files in current directory:
  - .venv
  - ACLED_egypt2011.csv
  - bella_copia.ipynb


The problem in this dataset is that there are arabic letters that python cannot read (therefore, using encoding=latin1 as suggested by AI resolves this issue). Moreover, going through the dataset is clear that some rows have issues in format that gave me 'parses error' a lot, so I decided to skip these few rows and continue loading the remaining valid data.  

In [23]:
import pandas as pd

# index_col=False prevents pandas from incorrectly treating leading columns as index
df = pd.read_csv('ACLED_egypt2011.csv', encoding='latin1', on_bad_lines='skip', index_col=False)
print(f"Dataset dimensions: {df.shape[0]:,} rows and {df.shape[1]} columns")

Dataset dimensions: 2,607 rows and 32 columns


In [75]:
# Count rows in the filtered ACLED_alsisi protest set where israel/israeli appears in any text column.
text_cols = df_alsisi_filtered.select_dtypes(include=['object', 'string']).columns
pattern = r'\bisrael(?:i)?\b'

row_text = df_alsisi_filtered[text_cols].fillna('').astype(str).agg(' '.join, axis=1)
israel_rows = row_text.str.contains(pattern, case=False, regex=True, na=False)

print(f"Filtered protest rows: {len(df_alsisi_filtered):,}")
print(f"Rows mentioning 'israel/israeli': {israel_rows.sum():,}")

Filtered protest rows: 1,685
Rows mentioning 'israel/israeli': 145


In [24]:
# Can you remove all the rows related to 'mob violence'?
df = df[df['event_type'] != 'Mob violence']
print(f"Dataset dimensions after removing 'mob violence': {df.shape[0]:,} rows and {df.shape[1]} columns")


Dataset dimensions after removing 'mob violence': 2,607 rows and 32 columns


In [27]:
import plotly.express as px

# Filter Egypt ACLED events across the full dataset and remove Mob violence before charting.
df_all = df.copy()
df_all['event_date'] = pd.to_datetime(df_all['event_date'], errors='coerce')
df_all = df_all[df_all['sub_event_type'] != 'Mob violence'].copy()
df_all['month'] = df_all['event_date'].dt.to_period('M').dt.to_timestamp()
df_all['month_label'] = df_all['month'].dt.strftime('%Y-%m')

month_order = (
    df_all[['month', 'month_label']]
    .drop_duplicates()
    .sort_values('month')
    ['month_label']
    .tolist()
)

monthly_sub_events = (
    df_all.groupby(['month_label', 'sub_event_type'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
)

fig = px.bar(
    monthly_sub_events,
    x='month_label',
    y='count',
    color='sub_event_type',
    category_orders={'month_label': month_order},
    color_discrete_map={
        'Excessive force against protesters': 'gray',
        'Protest with intervention': 'orange',
        'Violent demonstration': 'black',
        'Peaceful protest': 'red',
    },
    title='ACLED Egypt: Monthly events by sub-event type without Mob violence',
    labels={'month_label': 'Month', 'count': 'Event count', 'sub_event_type': 'Sub-event type'},
    barmode='stack',
    text_auto=True,
)

fig.update_layout(
    template='plotly_white',
    xaxis_title='Month',
    yaxis_title='Event count',
    legend_title='Sub-event type',
    bargap=0.15,
    height=600,
)

fig.show()

In [29]:
import plotly.express as px

# Synthesize the same filtered ACLED data by year instead of month.
df_yearly = df.copy()
df_yearly['event_date'] = pd.to_datetime(df_yearly['event_date'], errors='coerce')
df_yearly = df_yearly[df_yearly['sub_event_type'] != 'Mob violence'].copy()
df_yearly['year'] = df_yearly['event_date'].dt.year

year_order = (
    df_yearly[['year']]
    .dropna()
    .drop_duplicates()
    .sort_values('year')
    ['year']
    .astype(int)
    .tolist()
)

yearly_sub_events = (
    df_yearly.dropna(subset=['year'])
    .groupby(['year', 'sub_event_type'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
)

yearly_sub_events['year'] = yearly_sub_events['year'].astype(int)

fig_year = px.bar(
    yearly_sub_events,
    x='year',
    y='count',
    color='sub_event_type',
    category_orders={'year': year_order},
    color_discrete_map={
        'Excessive force against protesters': 'gray',
        'Protest with intervention': 'orange',
        'Violent demonstration': 'black',
        'Peaceful protest': 'red',
    },
    title='ACLED Egypt: Yearly events by sub-event type without Mob violence',
    labels={'year': 'Year', 'count': 'Event count', 'sub_event_type': 'Sub-event type'},
    barmode='stack',
    text_auto=True,
)

fig_year.update_layout(
    template='plotly_white',
    xaxis_title='Year',
    yaxis_title='Event count',
    legend_title='Sub-event type',
    bargap=0.15,
    height=600,
)

fig_year.show()

In [32]:
import plotly.express as px

# Map protest locations using exact coordinates, split into the four non-mob sub-event types.
norm_cols = {c.strip().lower(): c for c in df.columns}
pick = lambda *names: next((norm_cols[n] for n in names if n in norm_cols), None)

date_col = pick('event_date', 'date')
event_col = pick('event_type')
disorder_col = pick('disorder_type')
sub_event_col = pick('sub_event_type')
lat_col = pick('latitude', 'lat')
lon_col = pick('longitude', 'lon', 'lng')
loc_col = pick('location', 'admin1', 'admin2', 'admin3')

if not date_col or not lat_col or not lon_col or not sub_event_col:
    raise ValueError('Missing required columns (date, coordinates, and sub_event_type).')

allowed_sub_events = [
    'Excessive force against protesters',
    'Protest with intervention',
    'Violent demonstration',
    'Peaceful protest',
]

protest_map_df = df.copy()
protest_map_df[date_col] = pd.to_datetime(protest_map_df[date_col], errors='coerce')
protest_map_df[lat_col] = pd.to_numeric(protest_map_df[lat_col], errors='coerce')
protest_map_df[lon_col] = pd.to_numeric(protest_map_df[lon_col], errors='coerce')
protest_map_df = protest_map_df[
    protest_map_df[sub_event_col].isin(allowed_sub_events)
].copy()
protest_map_df = protest_map_df.dropna(subset=[date_col, lat_col, lon_col])

if protest_map_df.empty:
    print('No protest events found with valid date and coordinates.')
else:
    group_cols = [lat_col, lon_col, sub_event_col] + ([loc_col] if loc_col else [])
    protest_bubbles = (
        protest_map_df
        .groupby(group_cols, dropna=False)
        .size()
        .rename('protest_count')
        .reset_index()
    )

    fig_map = px.scatter_map(
        protest_bubbles,
        lat=lat_col,
        lon=lon_col,
        size='protest_count',
        color=sub_event_col,
        hover_name=loc_col if loc_col else None,
        hover_data=['protest_count'],
        zoom=5,
        height=700,
        size_max=40,
        title='Egypt Protest Density Map by Exact Coordinates and Sub-event Type',
        labels={
            'protest_count': 'Protest count',
            sub_event_col: 'Sub-event type',
        },
        color_discrete_map={
            'Excessive force against protesters': 'gray',
            'Protest with intervention': 'orange',
            'Violent demonstration': 'black',
            'Peaceful protest': 'red',
        },
    )
    fig_map.update_traces(marker=dict(opacity=0.7))
    fig_map.update_layout(margin=dict(l=0, r=0, t=50, b=0))
    fig_map.show()

print(f'Mapped protest events with coordinates: {len(protest_map_df):,}')

Mapped protest events with coordinates: 2,366


In [34]:
import plotly.express as px

# Timeline map: exact coordinates, four protest sub-event categories, no Mob violence.
norm_cols = {c.strip().lower(): c for c in df.columns}
pick = lambda *names: next((norm_cols[n] for n in names if n in norm_cols), None)

date_col = pick('event_date', 'date')
sub_event_col = pick('sub_event_type')
lat_col = pick('latitude', 'lat')
lon_col = pick('longitude', 'lon', 'lng')
loc_col = pick('location', 'admin1', 'admin2', 'admin3')

if not date_col or not lat_col or not lon_col or not sub_event_col:
    raise ValueError('Missing required columns (date, coordinates, and sub_event_type).')

allowed_sub_events = [
    'Excessive force against protesters',
    'Protest with intervention',
    'Violent demonstration',
    'Peaceful protest',
]

timeline_df = df.copy()
timeline_df[date_col] = pd.to_datetime(timeline_df[date_col], errors='coerce')
timeline_df[lat_col] = pd.to_numeric(timeline_df[lat_col], errors='coerce')
timeline_df[lon_col] = pd.to_numeric(timeline_df[lon_col], errors='coerce')
timeline_df = timeline_df[timeline_df[sub_event_col].isin(allowed_sub_events)].copy()
timeline_df = timeline_df.dropna(subset=[date_col, lat_col, lon_col])
timeline_df['month'] = timeline_df[date_col].dt.to_period('M').astype(str)

if timeline_df.empty:
    print('No protest events found with valid date and coordinates.')
else:
    group_cols = ['month', lat_col, lon_col, sub_event_col] + ([loc_col] if loc_col else [])
    timeline_bubbles = (
        timeline_df
        .groupby(group_cols, dropna=False)
        .size()
        .rename('protest_count')
        .reset_index()
    )

    fig_map_timeline = px.scatter_map(
        timeline_bubbles,
        lat=lat_col,
        lon=lon_col,
        size='protest_count',
        color=sub_event_col,
        animation_frame='month',
        hover_name=loc_col if loc_col else None,
        hover_data=['protest_count'],
        zoom=5,
        height=700,
        size_max=40,
        title='Egypt Protest Density Timeline by Exact Coordinates and Sub-event Type',
        labels={
            'protest_count': 'Protest count',
            sub_event_col: 'Sub-event type',
            'month': 'Month',
        },
        color_discrete_map={
            'Excessive force against protesters': 'gray',
            'Protest with intervention': 'orange',
            'Violent demonstration': 'black',
            'Peaceful protest': 'red',
        },
    )
    fig_map_timeline.update_traces(marker=dict(opacity=0.7))
    fig_map_timeline.update_layout(margin=dict(l=0, r=0, t=50, b=0))
    fig_map_timeline.show()

print(f'Mapped protest events with timeline: {len(timeline_df):,}')

Mapped protest events with timeline: 2,366


In [36]:
import plotly.express as px

# Monthly line chart: all protest sub-event categories combined into one total per month.
norm_cols = {c.strip().lower(): c for c in df.columns}
pick = lambda *names: next((norm_cols[n] for n in names if n in norm_cols), None)

date_col = pick('event_date', 'date')
sub_event_col = pick('sub_event_type')

if not date_col or not sub_event_col:
    raise ValueError('Missing required columns (event_date/date and sub_event_type).')

allowed_sub_events = [
    'Excessive force against protesters',
    'Protest with intervention',
    'Violent demonstration',
    'Peaceful protest',
]

monthly_df = df.copy()
monthly_df[date_col] = pd.to_datetime(monthly_df[date_col], errors='coerce')
monthly_df = monthly_df[monthly_df[sub_event_col].isin(allowed_sub_events)].copy()

monthly_totals = (
    monthly_df.dropna(subset=[date_col])
    .set_index(date_col)
    .resample('ME')
    .size()
    .rename('total_protests')
    .reset_index()
    .sort_values(by=date_col)
)

if monthly_totals.empty:
    print('No protest events found for monthly totals.')
else:
    fig_line = px.line(
        monthly_totals,
        x=date_col,
        y='total_protests',
        markers=True,
        title='Total Protests per Month (All Sub-event Types Combined)',
        labels={date_col: 'Month', 'total_protests': 'Number of protests'},
    )
    fig_line.update_traces(line=dict(color='black', width=3), marker=dict(size=7, color='red'))
    fig_line.update_layout(template='plotly_white', hovermode='x unified', height=550)
    fig_line.update_xaxes(rangeslider_visible=False)
    fig_line.show()

print(f"Total protest events included in line chart: {len(monthly_df):,}")

Total protest events included in line chart: 2,366


In [37]:
import plotly.express as px

# Percentage charts (100% basis): monthly and yearly for the four protest sub-event categories.
norm_cols = {c.strip().lower(): c for c in df.columns}
pick = lambda *names: next((norm_cols[n] for n in names if n in norm_cols), None)

date_col = pick('event_date', 'date')
sub_event_col = pick('sub_event_type')

if not date_col or not sub_event_col:
    raise ValueError('Missing required columns (event_date/date and sub_event_type).')

allowed_sub_events = [
    'Excessive force against protesters',
    'Protest with intervention',
    'Violent demonstration',
    'Peaceful protest',
]

pct_df = df.copy()
pct_df[date_col] = pd.to_datetime(pct_df[date_col], errors='coerce')
pct_df = pct_df[pct_df[sub_event_col].isin(allowed_sub_events)].copy()
pct_df = pct_df.dropna(subset=[date_col])

# Monthly percentage table
pct_df['month'] = pct_df[date_col].dt.to_period('M').astype(str)
monthly_counts = (
    pct_df.groupby(['month', sub_event_col], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
)
monthly_counts['percent'] = (
    monthly_counts['count']
    / monthly_counts.groupby('month')['count'].transform('sum')
    * 100
)

# Yearly percentage table
pct_df['year'] = pct_df[date_col].dt.year.astype(int)
yearly_counts = (
    pct_df.groupby(['year', sub_event_col], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
)
yearly_counts['percent'] = (
    yearly_counts['count']
    / yearly_counts.groupby('year')['count'].transform('sum')
    * 100
)

color_map = {
    'Excessive force against protesters': 'gray',
    'Protest with intervention': 'orange',
    'Violent demonstration': 'black',
    'Peaceful protest': 'red',
}

if monthly_counts.empty:
    print('No protest events found for monthly percentage chart.')
else:
    fig_month_pct = px.bar(
        monthly_counts,
        x='month',
        y='percent',
        color=sub_event_col,
        title='Monthly Protest Composition by Sub-event Type (100% Stacked)',
        labels={'month': 'Month', 'percent': 'Share (%)', sub_event_col: 'Sub-event type'},
        color_discrete_map=color_map,
        barmode='stack',
    )
    fig_month_pct.update_layout(template='plotly_white', hovermode='x unified', height=600)
    fig_month_pct.update_yaxes(title_text='Share (%)', range=[0, 100], ticksuffix='%')
    fig_month_pct.update_xaxes(title_text='Month')
    fig_month_pct.show()

if yearly_counts.empty:
    print('No protest events found for yearly percentage chart.')
else:
    fig_year_pct = px.bar(
        yearly_counts,
        x='year',
        y='percent',
        color=sub_event_col,
        title='Yearly Protest Composition by Sub-event Type (100% Stacked)',
        labels={'year': 'Year', 'percent': 'Share (%)', sub_event_col: 'Sub-event type'},
        color_discrete_map=color_map,
        barmode='stack',
    )
    fig_year_pct.update_layout(template='plotly_white', hovermode='x unified', height=600)
    fig_year_pct.update_yaxes(title_text='Share (%)', range=[0, 100], ticksuffix='%')
    fig_year_pct.update_xaxes(title_text='Year')
    fig_year_pct.show()

print(f"Total protest events used for percentage charts: {len(pct_df):,}")

Total protest events used for percentage charts: 2,366


In [106]:
import pandas as pd

# ACLED_alsisi: Sep 2019 to Sep 2020, without israel/palestine keyword filtering.
df_alsisi2 = pd.read_csv('ACLED_alsisi.csv', encoding='latin1', on_bad_lines='skip', index_col=False)
df_alsisi2['event_date'] = pd.to_datetime(df_alsisi2['event_date'], errors='coerce')
df_alsisi2 = df_alsisi2[
    (df_alsisi2['event_date'] >= '2019-09-01') &
    (df_alsisi2['event_date'] <= '2020-09-30')
].copy()

allowed_sub_events_alsisi2 = [
    'Excessive force against protesters',
    'Protest with intervention',
    'Violent demonstration',
    'Peaceful protest',
]
color_map_alsisi2 = {
    'Excessive force against protesters': 'gray',
    'Protest with intervention': 'orange',
    'Violent demonstration': 'black',
    'Peaceful protest': 'red',
}

df_alsisi2_filtered = df_alsisi2[df_alsisi2['sub_event_type'].isin(allowed_sub_events_alsisi2)].copy()
print(f"ACLED_alsisi2 rows in date window: {len(df_alsisi2):,}")
print(f"Rows in 4 protest categories: {len(df_alsisi2_filtered):,}")

ACLED_alsisi2 rows in date window: 153
Rows in 4 protest categories: 149


In [107]:
import plotly.express as px

# 1) Monthly and yearly stacked counts by sub-event type.
alsisi2_monthly = df_alsisi2_filtered.dropna(subset=['event_date']).copy()
alsisi2_monthly['month'] = alsisi2_monthly['event_date'].dt.to_period('M').astype(str)
alsisi2_monthly_counts = (
    alsisi2_monthly.groupby(['month', 'sub_event_type'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
)

fig_alsisi2_month = px.bar(
    alsisi2_monthly_counts,
    x='month',
    y='count',
    color='sub_event_type',
    color_discrete_map=color_map_alsisi2,
    barmode='stack',
    title='ACLED_alsisi: Monthly Events by Sub-event Type (Sep 2019 to Sep 2020)',
    labels={'month': 'Month', 'count': 'Event count', 'sub_event_type': 'Sub-event type'},
)
fig_alsisi2_month.update_layout(template='plotly_white', hovermode='x unified', height=600)
fig_alsisi2_month.show()

alsisi2_yearly = df_alsisi2_filtered.dropna(subset=['event_date']).copy()
alsisi2_yearly['year'] = alsisi2_yearly['event_date'].dt.year.astype(int)
alsisi2_yearly_counts = (
    alsisi2_yearly.groupby(['year', 'sub_event_type'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
)

fig_alsisi2_year = px.bar(
    alsisi2_yearly_counts,
    x='year',
    y='count',
    color='sub_event_type',
    color_discrete_map=color_map_alsisi2,
    barmode='stack',
    title='ACLED_alsisi: Yearly Events by Sub-event Type (Sep 2019 to Sep 2020)',
    labels={'year': 'Year', 'count': 'Event count', 'sub_event_type': 'Sub-event type'},
)
fig_alsisi2_year.update_layout(template='plotly_white', hovermode='x unified', height=600)
fig_alsisi2_year.show()

In [111]:
import plotly.express as px

# 2) Exact-coordinate bubble map and timeline map.
alsisi2_map = df_alsisi2_filtered.dropna(subset=['event_date', 'latitude', 'longitude']).copy()
alsisi2_map['latitude'] = pd.to_numeric(alsisi2_map['latitude'], errors='coerce')
alsisi2_map['longitude'] = pd.to_numeric(alsisi2_map['longitude'], errors='coerce')
alsisi2_map = alsisi2_map.dropna(subset=['latitude', 'longitude'])

alsisi2_bubbles = (
    alsisi2_map.groupby(['latitude', 'longitude', 'sub_event_type', 'location'], dropna=False)
    .size()
    .rename('protest_count')
    .reset_index()
)

fig_alsisi2_map = px.scatter_map(
    alsisi2_bubbles,
    lat='latitude',
    lon='longitude',
    size='protest_count',
    color='sub_event_type',
    color_discrete_map=color_map_alsisi2,
    hover_name='location',
    hover_data=['protest_count'],
    zoom=5,
    height=700,
    size_max=40,
    title='ACLED_alsisi: Protest Density by Exact Coordinates (Sep 2019 to Sep 2020)',
)
fig_alsisi2_map.update_traces(marker=dict(opacity=0.7))
fig_alsisi2_map.update_layout(margin=dict(l=0, r=0, t=50, b=0))
fig_alsisi2_map.show()

alsisi2_timeline = alsisi2_map.copy()
alsisi2_timeline['month'] = alsisi2_timeline['event_date'].dt.to_period('M').astype(str)
alsisi2_timeline_bubbles = (
    alsisi2_timeline.groupby(['month', 'latitude', 'longitude', 'sub_event_type', 'location'], dropna=False)
    .size()
    .rename('protest_count')
    .reset_index()
)

fig_alsisi2_timeline = px.scatter_map(
    alsisi2_timeline_bubbles,
    lat='latitude',
    lon='longitude',
    size='protest_count',
    color='sub_event_type',
    color_discrete_map=color_map_alsisi2,
    animation_frame='month',
    hover_name='location',
    hover_data=['protest_count'],
    zoom=5,
    height=700,
    size_max=40,
    title='ACLED_alsisi: Protest Density Timeline (Sep 2019 to Sep 2020)',
)
fig_alsisi2_timeline.update_traces(marker=dict(opacity=0.7))
fig_alsisi2_timeline.update_layout(margin=dict(l=0, r=0, t=50, b=0))
fig_alsisi2_timeline.show()

In [112]:
import plotly.express as px

# 3) Monthly total line chart.
alsisi2_line = df_alsisi2_filtered.dropna(subset=['event_date']).copy()
alsisi2_line_totals = (
    alsisi2_line.set_index('event_date')
    .resample('ME')
    .size()
    .rename('total_protests')
    .reset_index()
)

fig_alsisi2_line = px.line(
    alsisi2_line_totals,
    x='event_date',
    y='total_protests',
    markers=True,
    title='ACLED_alsisi: Total Protests per Month (Sep 2019 to Sep 2020)',
    labels={'event_date': 'Month', 'total_protests': 'Number of protests'},
)
fig_alsisi2_line.update_traces(line=dict(color='black', width=3), marker=dict(size=7, color='red'))
fig_alsisi2_line.update_layout(template='plotly_white', hovermode='x unified', height=550)
fig_alsisi2_line.update_xaxes(rangeslider_visible=False)
fig_alsisi2_line.show()

# 4) 100% stacked monthly and yearly composition charts.
alsisi2_pct = df_alsisi2_filtered.dropna(subset=['event_date']).copy()
alsisi2_pct['month'] = alsisi2_pct['event_date'].dt.to_period('M').astype(str)
alsisi2_month_pct = (
    alsisi2_pct.groupby(['month', 'sub_event_type'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
)
alsisi2_month_pct['percent'] = (
    alsisi2_month_pct['count'] / alsisi2_month_pct.groupby('month')['count'].transform('sum') * 100
)

fig_alsisi2_month_pct = px.bar(
    alsisi2_month_pct,
    x='month',
    y='percent',
    color='sub_event_type',
    color_discrete_map=color_map_alsisi2,
    barmode='stack',
    title='ACLED_alsisi: Monthly Composition by Sub-event Type (100% Stacked)',
    labels={'month': 'Month', 'percent': 'Share (%)', 'sub_event_type': 'Sub-event type'},
)
fig_alsisi2_month_pct.update_layout(template='plotly_white', hovermode='x unified', height=600)
fig_alsisi2_month_pct.update_yaxes(title_text='Share (%)', range=[0, 100], ticksuffix='%')
fig_alsisi2_month_pct.show()

alsisi2_pct['year'] = alsisi2_pct['event_date'].dt.year.astype(int)
alsisi2_year_pct = (
    alsisi2_pct.groupby(['year', 'sub_event_type'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
)
alsisi2_year_pct['percent'] = (
    alsisi2_year_pct['count'] / alsisi2_year_pct.groupby('year')['count'].transform('sum') * 100
)

fig_alsisi2_year_pct = px.bar(
    alsisi2_year_pct,
    x='year',
    y='percent',
    color='sub_event_type',
    color_discrete_map=color_map_alsisi2,
    barmode='stack',
    title='ACLED_alsisi: Yearly Composition by Sub-event Type (100% Stacked)',
    labels={'year': 'Year', 'percent': 'Share (%)', 'sub_event_type': 'Sub-event type'},
)
fig_alsisi2_year_pct.update_layout(template='plotly_white', hovermode='x unified', height=600)
fig_alsisi2_year_pct.update_yaxes(title_text='Share (%)', range=[0, 100], ticksuffix='%')
fig_alsisi2_year_pct.show()

print(f"Total ACLED_alsisi2 protest events used in charts: {len(df_alsisi2_filtered):,}")

Total ACLED_alsisi2 protest events used in charts: 149
